# Premier League Probability Engine

## Phase 2: Feature Engineering

The Elo model developed in Notebook 1 provides a baseline measure of team strength.

This notebook focuses on constructing the features that will be available before each match. These features will later be used to predict match outcomes and compare the model's probabilities with bookmaker odds.

The key principle throughout this notebook is to avoid look-ahead bias. Every feature must be calculated using only information that would have been known before kickoff.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

## 1. Loading multiple Premier League seasons

The first Elo model used only the 2024–25 season.

For predictive modelling, several seasons are needed so that team ratings and rolling features have enough historical information. This also means teams do not all begin the target season with identical ratings.

The code below loads Premier League match data from 2015–16 through 2024–25 and combines the seasons into one chronological dataset.

In [2]:
season_codes = {
    "2015-16": "1516",
    "2016-17": "1617",
    "2017-18": "1718",
    "2018-19": "1819",
    "2019-20": "1920",
    "2020-21": "2021",
    "2021-22": "2122",
    "2022-23": "2223",
    "2023-24": "2324",
    "2024-25": "2425",
}

season_frames = []

for season_name, season_code in season_codes.items():
    url = (
        "https://www.football-data.co.uk/mmz4281/"
        f"{season_code}/E0.csv"
    )

    season_data = pd.read_csv(url)

    season_data["Season"] = season_name

    season_frames.append(season_data)

matches_all = pd.concat(
    season_frames,
    ignore_index=True,
)

print(f"Seasons loaded: {len(season_frames)}")
print(f"Total matches: {len(matches_all)}")

matches_all[["Season", "Date", "HomeTeam", "AwayTeam"]].head()

Seasons loaded: 10
Total matches: 3800


C:\Users\kiera\AppData\Local\Temp\ipykernel_6844\2526682352.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  season_data["Season"] = season_name


,Season,Date,HomeTeam,AwayTeam
0,2015-16,08/08/2015,Bournemouth,Aston Villa
1,2015-16,08/08/2015,Chelsea,Swansea
2,2015-16,08/08/2015,Everton,Watford
3,2015-16,08/08/2015,Leicester,Sunderland
4,2015-16,08/08/2015,Man United,Tottenham


## 2. Checking column consistency

Before combining data from different seasons, it is important to confirm that the files use compatible column names.

Football-data files sometimes gain or lose bookmaker columns between seasons. The core match columns should remain consistent, but this must be checked rather than assumed.

In [3]:
column_counts = {}

for season_name, season_code in season_codes.items():
    url = (
        "https://www.football-data.co.uk/mmz4281/"
        f"{season_code}/E0.csv"
    )

    season_data = pd.read_csv(url)

    column_counts[season_name] = len(season_data.columns)

pd.Series(column_counts, name="Number of columns")

2015-16     65
2016-17     65
2017-18     65
2018-19     62
2019-20    106
2020-21    106
2021-22    106
2022-23    106
2023-24    106
2024-25    120
Name: Number of columns, dtype: int64

In [4]:
required_columns = [
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
]

missing_columns = {}

for season_name, season_code in season_codes.items():
    url = (
        "https://www.football-data.co.uk/mmz4281/"
        f"{season_code}/E0.csv"
    )

    season_data = pd.read_csv(url)

    missing = [
        column
        for column in required_columns
        if column not in season_data.columns
    ]

    missing_columns[season_name] = missing

missing_columns

{'2015-16': [],
 '2016-17': [],
 '2017-18': [],
 '2018-19': [],
 '2019-20': [],
 '2020-21': [],
 '2021-22': [],
 '2022-23': [],
 '2023-24': [],
 '2024-25': []}

## 3. Selecting the core match columns

The raw files contain many bookmaker and match-statistics columns that vary between seasons.

For the first feature-engineering stage, only the columns needed to identify each match and its final result are retained. Additional variables can be added later when they have a clear modelling purpose.

In [5]:
core_columns = [
    "Season",
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
]

matches = matches_all[core_columns].copy()

matches.head()

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,2015-16,08/08/2015,Bournemouth,Aston Villa,0,1,A
1,2015-16,08/08/2015,Chelsea,Swansea,2,2,D
2,2015-16,08/08/2015,Everton,Watford,2,2,D
3,2015-16,08/08/2015,Leicester,Sunderland,4,2,H
4,2015-16,08/08/2015,Man United,Tottenham,1,0,H


In [6]:
print(f"Rows: {matches.shape[0]}")
print(f"Columns: {matches.shape[1]}")

Rows: 3800
Columns: 7


In [7]:
matches.isna().sum()

Season      0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
dtype: int64

## 4. Converting dates and ordering matches

All predictive features must be calculated using information available before each match.

The matches are therefore converted into proper datetime values and sorted chronologically. This ensures that rolling statistics and Elo ratings are updated in the correct order and prevents look-ahead bias.

In [8]:
matches["Date"] = pd.to_datetime(
    matches["Date"],
    dayfirst=True,
    errors="coerce",
)

matches["Date"].isna().sum()

np.int64(380)

In [9]:
matches = (
    matches
    .sort_values(["Date", "HomeTeam", "AwayTeam"])
    .reset_index(drop=True)
)

matches[["Season", "Date", "HomeTeam", "AwayTeam"]].head(10)

,Season,Date,HomeTeam,AwayTeam
0,2015-16,2015-08-08,Bournemouth,Aston Villa
1,2015-16,2015-08-08,Chelsea,Swansea
2,2015-16,2015-08-08,Everton,Watford
3,2015-16,2015-08-08,Leicester,Sunderland
4,2015-16,2015-08-08,Man United,Tottenham
5,2015-16,2015-08-08,Norwich,Crystal Palace
6,2015-16,2015-08-09,Arsenal,West Ham
7,2015-16,2015-08-09,Newcastle,Southampton
8,2015-16,2015-08-09,Stoke,Liverpool
9,2015-16,2015-08-10,West Brom,Man City


In [10]:
print("First match date:", matches["Date"].min())
print("Last match date:", matches["Date"].max())

First match date: 2015-08-08 00:00:00
Last match date: 2025-05-25 00:00:00


## 5. Creating pre-match Elo features

The first engineered features will represent each team's estimated strength immediately before kickoff.

To avoid target leakage, the rating used as a feature must be recorded before the result of the current match is used to update the Elo system. Each match will therefore follow this sequence:

1. Retrieve the current home and away ratings.
2. Store those ratings as pre-match features.
3. Calculate the expected match scores.
4. Observe the match result.
5. Update both teams' ratings.

The Elo parameters are kept consistent with the baseline model developed in Notebook 1:

- Initial team rating: $1500$
- Update factor: $K=20$
- Home-advantage adjustment: $H=50$

For the initial implementation, ratings will carry continuously across season boundaries. This preserves information about team strength between adjacent seasons. Seasonal rating regression can later be investigated as a modelling extension.

In [13]:
# Elo model parameters
initial_rating = 1500
k_factor = 20
home_advantage = 50


def expected_score(rating_a, rating_b):
    """
    Calculate Team A's expected score against Team B.

    Parameters
    ----------
    rating_a : float
        Elo rating of Team A, including any applicable adjustment.
    rating_b : float
        Elo rating of Team B.

    Returns
    -------
    float
        Expected score between 0 and 1.
    """
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def actual_scores_from_goals(home_goals, away_goals):
    """
    Convert the match score into Elo actual scores.

    Returns
    -------
    tuple
        Home and away actual scores.
    """
    if home_goals > away_goals:
        return 1.0, 0.0

    if home_goals < away_goals:
        return 0.0, 1.0

    return 0.5, 0.5


def update_elo_ratings(
    home_rating,
    away_rating,
    home_goals,
    away_goals,
):
    """
    Calculate expected scores and update both teams' Elo ratings.
    """
    home_expected = expected_score(
        home_rating + home_advantage,
        away_rating,
    )
    away_expected = 1 - home_expected

    home_actual, away_actual = actual_scores_from_goals(
        home_goals,
        away_goals,
    )

    updated_home_rating = (
        home_rating
        + k_factor * (home_actual - home_expected)
    )

    updated_away_rating = (
        away_rating
        + k_factor * (away_actual - away_expected)
    )

    return (
        updated_home_rating,
        updated_away_rating,
        home_expected,
        away_expected,
    )

### 5.1 Validating the Elo functions

Before processing the full dataset, the implementation is tested using two equally rated teams.

Because the home team receives a 50-point rating adjustment, its expected score should be greater than $0.5$. If the home team then wins, its rating should increase and the away team's rating should decrease by the same amount.

This check helps confirm that home advantage is being applied in the correct direction and that the Elo update remains zero-sum.

In [14]:
test_result = update_elo_ratings(
    home_rating=initial_rating,
    away_rating=initial_rating,
    home_goals=2,
    away_goals=1,
)

(
    test_home_rating,
    test_away_rating,
    test_home_expected,
    test_away_expected,
) = test_result

print(f"Home expected score: {test_home_expected:.4f}")
print(f"Away expected score: {test_away_expected:.4f}")
print(f"Updated home rating: {test_home_rating:.2f}")
print(f"Updated away rating: {test_away_rating:.2f}")

points_conserved = np.isclose(
    test_home_rating + test_away_rating,
    2 * initial_rating,
)

print(f"Rating points conserved: {points_conserved}")

Home expected score: 0.5715
Away expected score: 0.4285
Updated home rating: 1508.57
Updated away rating: 1491.43
Rating points conserved: True


### 5.2 Initialising the Elo rating state

The Elo model is sequential: the rating available before a match depends only on matches that occurred previously.

A dictionary called `current_ratings` will store the latest available rating for every team. When a team appears for the first time in the dataset, it will be assigned the common initial rating of $1500$.

Two lists will store the ratings observed immediately before each fixture:

- `home_elo_before`
- `away_elo_before`

These lists will follow the same chronological row order as the `matches` DataFrame. After the full sequence has been processed, they can be added as pre-match feature columns.

At this stage, the objects are only initialised. No match outcomes have yet been processed.

In [15]:
# Store each team's latest Elo rating
current_ratings = {}

# Store ratings observed before each match
home_elo_before = []
away_elo_before = []

print(f"Teams currently rated: {len(current_ratings)}")
print(f"Home Elo observations stored: {len(home_elo_before)}")
print(f"Away Elo observations stored: {len(away_elo_before)}")

Teams currently rated: 0
Home Elo observations stored: 0
Away Elo observations stored: 0


### 5.3 Processing a single match

Before constructing the full chronological loop, the first match is processed manually.

For each fixture, the Elo feature-generation sequence must be:

1. Retrieve the ratings available before kickoff.
2. Store those ratings as the match features.
3. Calculate the expected scores.
4. Use the observed result to update both ratings.
5. Store the updated ratings for future fixtures.

This ordering prevents target leakage. The current match result is allowed to influence future ratings, but it must not influence the features recorded for the current match.

In [16]:
# Select the first match in the chronologically sorted dataset
first_match = matches.iloc[0]

home_team = first_match["HomeTeam"]
away_team = first_match["AwayTeam"]

home_goals = first_match["FTHG"]
away_goals = first_match["FTAG"]

# Assign the initial rating when a team appears for the first time
home_rating_before = current_ratings.get(
    home_team,
    initial_rating,
)

away_rating_before = current_ratings.get(
    away_team,
    initial_rating,
)

# Store the ratings before using the match result
home_elo_before.append(home_rating_before)
away_elo_before.append(away_rating_before)

# Update the ratings using the observed result
(
    updated_home_rating,
    updated_away_rating,
    home_expected,
    away_expected,
) = update_elo_ratings(
    home_rating=home_rating_before,
    away_rating=away_rating_before,
    home_goals=home_goals,
    away_goals=away_goals,
)

# Save the updated ratings for future matches
current_ratings[home_team] = updated_home_rating
current_ratings[away_team] = updated_away_rating

print(f"Fixture: {home_team} {home_goals}-{away_goals} {away_team}")
print(f"Home Elo before kickoff: {home_rating_before:.2f}")
print(f"Away Elo before kickoff: {away_rating_before:.2f}")
print(f"Home expected score: {home_expected:.4f}")
print(f"Away expected score: {away_expected:.4f}")
print(f"Updated {home_team} rating: {updated_home_rating:.2f}")
print(f"Updated {away_team} rating: {updated_away_rating:.2f}")

Fixture: Bournemouth 0-1 Aston Villa
Home Elo before kickoff: 1500.00
Away Elo before kickoff: 1500.00
Home expected score: 0.5715
Away expected score: 0.4285
Updated Bournemouth rating: 1488.57
Updated Aston Villa rating: 1511.43


#### Verifying the state after one match

After one match:

- exactly one home pre-match rating should have been stored;
- exactly one away pre-match rating should have been stored;
- exactly two teams should exist in the ratings dictionary;
- the stored pre-match ratings should both equal $1500$.

These checks confirm that the feature values were recorded before the current result altered the Elo state.

In [17]:
assert len(home_elo_before) == 1
assert len(away_elo_before) == 1
assert len(current_ratings) == 2

assert home_elo_before[0] == initial_rating
assert away_elo_before[0] == initial_rating

assert np.isclose(
    updated_home_rating + updated_away_rating,
    2 * initial_rating,
)

print("Single-match Elo processing checks passed.")

Single-match Elo processing checks passed.


### 5.4 Resetting the Elo state

The first match was processed manually only to validate the sequential Elo logic.

Before applying the model to the complete dataset, the temporary state must be reset. Otherwise, the first fixture would be processed twice and all later ratings would be incorrect.

The ratings dictionary and pre-match feature lists are therefore returned to their original empty states. The Elo parameters and helper functions remain unchanged.

In [18]:
# Reset the temporary state used in the single-match test
current_ratings = {}

home_elo_before = []
away_elo_before = []

assert len(current_ratings) == 0
assert len(home_elo_before) == 0
assert len(away_elo_before) == 0

print("Elo state successfully reset.")

Elo state successfully reset.


### 5.5 Generating Elo features across the full match history

The Elo system is now applied sequentially to all 3,800 matches.

For every fixture, the ratings available immediately before kickoff are appended to the feature lists. Only after these values have been stored is the observed result used to update the teams' ratings.

This produces a path-dependent feature set in which each match uses only information that would genuinely have been available at that point in time.

Teams are assigned the initial rating of $1500$ when they first appear in the dataset. Updated ratings are then carried forward continuously across later fixtures and season boundaries.

In [19]:
# Process every match in chronological order
for match in matches.itertuples(index=False):

    home_team = match.HomeTeam
    away_team = match.AwayTeam

    home_goals = match.FTHG
    away_goals = match.FTAG

    # Retrieve ratings available before kickoff
    home_rating_before = current_ratings.get(
        home_team,
        initial_rating,
    )

    away_rating_before = current_ratings.get(
        away_team,
        initial_rating,
    )

    # Store pre-match ratings before using the current result
    home_elo_before.append(home_rating_before)
    away_elo_before.append(away_rating_before)

    # Update ratings using the observed result
    (
        updated_home_rating,
        updated_away_rating,
        home_expected,
        away_expected,
    ) = update_elo_ratings(
        home_rating=home_rating_before,
        away_rating=away_rating_before,
        home_goals=home_goals,
        away_goals=away_goals,
    )

    # Carry the updated ratings into future fixtures
    current_ratings[home_team] = updated_home_rating
    current_ratings[away_team] = updated_away_rating


# Validate the completed feature-generation process
assert len(home_elo_before) == len(matches)
assert len(away_elo_before) == len(matches)

assert not pd.Series(home_elo_before).isna().any()
assert not pd.Series(away_elo_before).isna().any()

print(f"Matches processed: {len(home_elo_before)}")
print(f"Teams rated: {len(current_ratings)}")
print("Full chronological Elo loop completed successfully.")

Matches processed: 3800
Teams rated: 34
Full chronological Elo loop completed successfully.


### 5.6 Adding the pre-match Elo features

The chronological Elo calculations are currently stored in two ordered lists.

Because the lists were generated by processing the `matches` DataFrame from its first row to its last row, each list position corresponds to the same match row in the DataFrame.

The following columns are added:

- `HomeEloBefore`: the home team's Elo rating immediately before kickoff;
- `AwayEloBefore`: the away team's Elo rating immediately before kickoff.

These are valid pre-match features because each value was stored before the current match result was used to update the Elo system.

In [21]:
# Attach the generated pre-match Elo features
matches["HomeEloBefore"] = home_elo_before
matches["AwayEloBefore"] = away_elo_before

# Confirm that the columns were created correctly
assert "HomeEloBefore" in matches.columns
assert "AwayEloBefore" in matches.columns

assert matches["HomeEloBefore"].notna().all()
assert matches["AwayEloBefore"].notna().all()

matches[
    [
        "Season",
        "Date",
        "HomeTeam",
        "AwayTeam",
        "FTHG",
        "FTAG",
        "HomeEloBefore",
        "AwayEloBefore",
    ]
].head(10)

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,HomeEloBefore,AwayEloBefore
0,2015-16,2015-08-08,Bournemouth,Aston Villa,0,1,1500.0,1500.0
1,2015-16,2015-08-08,Chelsea,Swansea,2,2,1500.0,1500.0
2,2015-16,2015-08-08,Everton,Watford,2,2,1500.0,1500.0
3,2015-16,2015-08-08,Leicester,Sunderland,4,2,1500.0,1500.0
4,2015-16,2015-08-08,Man United,Tottenham,1,0,1500.0,1500.0
5,2015-16,2015-08-08,Norwich,Crystal Palace,1,3,1500.0,1500.0
6,2015-16,2015-08-09,Arsenal,West Ham,0,2,1500.0,1500.0
7,2015-16,2015-08-09,Newcastle,Southampton,2,2,1500.0,1500.0
8,2015-16,2015-08-09,Stoke,Liverpool,0,1,1500.0,1500.0
9,2015-16,2015-08-10,West Brom,Man City,0,3,1500.0,1500.0


#### Checking the completed Elo feature columns

The completed feature columns should contain exactly one numerical value for every match.

Their distributions are also inspected to confirm that the ratings vary over time rather than remaining fixed at the initial value of $1500$.

In [22]:
elo_feature_summary = matches[
    ["HomeEloBefore", "AwayEloBefore"]
].describe()

print(elo_feature_summary)

assert len(matches) == 3800
assert matches["HomeEloBefore"].nunique() > 1
assert matches["AwayEloBefore"].nunique() > 1

print("\nPre-match Elo features added successfully.")

       HomeEloBefore  AwayEloBefore
count    3800.000000    3800.000000
mean     1533.494181    1532.357890
std        98.902603      99.602584
min      1313.084201    1309.475683
25%      1466.039750    1466.798872
50%      1511.452296    1509.482911
75%      1590.206040    1589.013806
max      1834.168087    1833.864724

Pre-match Elo features added successfully.


### 5.7 Auditing point-in-time correctness

The existence of the Elo columns does not by itself prove that the sequential state was handled correctly.

A direct point-in-time audit is therefore performed for one team. The team's rating after its first appearance is calculated independently and compared with the stored pre-match rating from its second appearance.

For a correctly implemented sequential model,

$$
R_{i,2}^{\text{before}}
=
R_{i,1}^{\text{after}}.
$$

Equality confirms that the first match result influenced the team's next match feature, but did not influence the feature recorded for the first match itself.

In [23]:
# Select a team automatically from the first chronological fixture
audit_team = matches.iloc[0]["HomeTeam"]

# Extract the team's first two appearances
audit_matches = matches.loc[
    (matches["HomeTeam"] == audit_team)
    | (matches["AwayTeam"] == audit_team)
].head(2).copy()

assert len(audit_matches) == 2

# Identify whether the team was playing at home or away
audit_matches["Venue"] = np.where(
    audit_matches["HomeTeam"] == audit_team,
    "Home",
    "Away",
)

# Extract the team's own pre-match Elo rating
audit_matches["TeamEloBefore"] = np.where(
    audit_matches["HomeTeam"] == audit_team,
    audit_matches["HomeEloBefore"],
    audit_matches["AwayEloBefore"],
)

first_appearance = audit_matches.iloc[0]
second_appearance = audit_matches.iloc[1]

# Independently reproduce the Elo update from the first appearance
(
    first_updated_home_rating,
    first_updated_away_rating,
    _,
    _,
) = update_elo_ratings(
    home_rating=first_appearance["HomeEloBefore"],
    away_rating=first_appearance["AwayEloBefore"],
    home_goals=first_appearance["FTHG"],
    away_goals=first_appearance["FTAG"],
)

# Select the updated rating belonging to the audited team
if first_appearance["HomeTeam"] == audit_team:
    calculated_rating_after_first_match = first_updated_home_rating
else:
    calculated_rating_after_first_match = first_updated_away_rating

stored_rating_before_second_match = second_appearance["TeamEloBefore"]

audit_passed = np.isclose(
    calculated_rating_after_first_match,
    stored_rating_before_second_match,
)

display(
    audit_matches[
        [
            "Date",
            "HomeTeam",
            "AwayTeam",
            "FTHG",
            "FTAG",
            "Venue",
            "TeamEloBefore",
        ]
    ]
)

print(f"Audited team: {audit_team}")
print(
    "Calculated rating after first match: "
    f"{calculated_rating_after_first_match:.6f}"
)
print(
    "Stored rating before second match: "
    f"{stored_rating_before_second_match:.6f}"
)
print(f"Point-in-time audit passed: {audit_passed}")

assert audit_passed

,Date,HomeTeam,AwayTeam,FTHG,FTAG,Venue,TeamEloBefore
0,2015-08-08,Bournemouth,Aston Villa,0,1,Home,1500.000000
19,2015-08-17,Liverpool,Bournemouth,1,0,Away,1488.570738


Audited team: Bournemouth
Calculated rating after first match: 1488.570738
Stored rating before second match: 1488.570738
Point-in-time audit passed: True
